In [1]:
import pandas as pd
import numpy as np
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv("smartphones_dataset.csv")

In [ ]:
categorical_features = ['Brand']
numerical_features = ['Screen_Size_in', 'Front_Camera_MP', 'Back_Camera_MP', 'Battery_mAh', 'RAM_GB', 'ROM_GB', 'Clock_Speed_Ghz']
target = 'Price_Rs'

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

In [ ]:
X = df[numerical_features + categorical_features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Screen_Size_in',
                                                   'Front_Camera_MP',
                                                   'Back_Camera_MP',
                                                   'Battery_mAh', 'RAM_GB',
                                                   'ROM_GB',
                                                   'Clock_Speed_Ghz']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Brand'])])),
                ('regressor', LinearRegression())])

In [ ]:
def create_ranges(min_val, max_val, step):
    ranges = []
    val = min_val
    while val < max_val:
        upper = min(val + step, max_val)
        ranges.append(f"{val}-{upper}")
        val = upper
    return ranges

In [ ]:
screen_sizes = create_ranges(df['Screen_Size_in'].min(), df['Screen_Size_in'].max(), 0.5)
front_cameras = create_ranges(df['Front_Camera_MP'].min(), df['Front_Camera_MP'].max(), 5)
back_cameras = create_ranges(df['Back_Camera_MP'].min(), df['Back_Camera_MP'].max(), 12)
batteries = create_ranges(df['Battery_mAh'].min(), df['Battery_mAh'].max(), 1000)
rams = create_ranges(df['RAM_GB'].min(), df['RAM_GB'].max(), 2)
roms = create_ranges(df['ROM_GB'].min(), df['ROM_GB'].max(), 64)
clock_speeds = create_ranges(df['Clock_Speed_Ghz'].min(), df['Clock_Speed_Ghz'].max(), 0.5)


In [ ]:
brands = df['Brand'].unique().tolist()
screen_sizes = ['5.0-5.5', '5.5-6.0', '6.0-6.5', '6.5-7.0', '7.0-7.5']
front_cameras = ['2-5', '5-8', '8-12', '12-16', '16-32', '32-42', '42-64']
back_cameras = ['8-12', '12-24', '24-48', '48-64', '64-108', '108-144', '144-160', '160-200']
batteries = ['2000-3000', '3000-4000', '4000-5000', '5000-5500', '5500-6000', '6000-6500']
rams = ['2-4', '4-6', '6-8', '8-12', '12-16', '16-20', '20-24']
roms = ['32-64', '64-128', '128-256', '256-512', '512-1024']
clock_speeds = ['2.0-2.5', '2.5-3.0', '3.0-3.5', '3.5-4.0', '4.0-4.5']

In [ ]:
def predict_price(brand, screen, front, back, battery, ram, rom, clock):

    input_data = pd.DataFrame([{
        'Brand': brand,
        'Screen_Size_in': range_to_mean(screen),
        'Front_Camera_MP': range_to_mean(front),
        'Back_Camera_MP': range_to_mean(back),
        'Battery_mAh': range_to_mean(battery),
        'RAM_GB': range_to_mean(ram),
        'ROM_GB': range_to_mean(rom),
        'Clock_Speed_Ghz': range_to_mean(clock)
    }])

    price = model.predict(input_data)[0]
    return f"₹ {price:,.0f}"

In [ ]:
iface = gr.Interface(
    fn=predict_price,
    inputs=[
        gr.Dropdown(choices=df['Brand'].unique().tolist(), label="Brand"),
        gr.Dropdown(choices=screen_sizes, label="Screen Size (inches)"),
        gr.Dropdown(choices=front_cameras, label="Front Camera (MP)"),
        gr.Dropdown(choices=back_cameras, label="Back Camera (MP)"),
        gr.Dropdown(choices=batteries, label="Battery (mAh)"),
        gr.Dropdown(choices=rams, label="RAM (GB)"),
        gr.Dropdown(choices=roms, label="ROM (GB)"),
        gr.Dropdown(choices=clock_speeds, label="Clock Speed (GHz)")
    ],
    outputs=gr.Textbox(label="Predicted Price"),
    title="Mobile Price Prediction"
)

In [ ]:
import pickle

with open("mobile_price_model.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
